# Module C: Circuit DNA & Driver Affinity

**F1 Race Intelligence Project**

Not all drivers perform equally at all circuits. Some tracks favor certain driving styles,
car setups, or team philosophies. This module quantifies these relationships through a
**driver-circuit affinity score** that combines finishing position, qualifying performance,
race-day positions gained, and reliability.

**Analyses performed:**
1. Driver-circuit affinity heatmap for top drivers
2. Top circuits per driver (ranked bar chart)
3. Circuit type classification and performance patterns
4. Hamilton at Silverstone: a deep-dive case study
5. Circuit characteristic profiles

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from src.viz import (
    affinity_heatmap, top_circuits_bar, f1_layout,
    F1_RED, F1_WHITE, F1_BG, F1_PAPER, TEAM_COLORS
)
from src.features import compute_affinity_score

DB_PATH = '../data/processed/f1.db'
con = duckdb.connect(DB_PATH, read_only=True)
print('Connected to DuckDB')

## 1. Driver-Circuit Affinity Overview

The affinity score (0-100) is a weighted composite of:
- **40%** — Finishing position percentile rank at the circuit
- **30%** — Qualifying closeness to pole position
- **20%** — Average positions gained during the race
- **10%** — Reliability (inverse DNF rate)

Higher scores indicate a stronger driver-circuit combination.

In [ ]:
# Load affinity data
affinity_df = con.execute("""
    SELECT *
    FROM driver_circuit_affinity
    WHERE affinity_score IS NOT NULL
    ORDER BY affinity_score DESC
""").fetchdf()

print(f'Total driver-circuit pairs: {len(affinity_df)}')
print(f'Unique drivers: {affinity_df["driver_id"].nunique()}')
print(f'Unique circuits: {affinity_df["circuit_id"].nunique()}')
print(f'\nAffinity score distribution:')
print(affinity_df['affinity_score'].describe().round(2))

print('\nTop 15 driver-circuit combinations:')
display(affinity_df[['driver_id', 'circuit_id', 'affinity_score', 'total_appearances', 'avg_finish_position']].head(15))

## 2. Affinity Heatmap: Top Drivers

A color-coded grid showing affinity scores for the current grid's top drivers
across major circuits. Green indicates strong affinity, red indicates weakness.

In [ ]:
# Select top drivers (2024 season regulars with sufficient historical data)
top_drivers = [
    'max_verstappen', 'hamilton', 'leclerc', 'norris', 'sainz',
    'perez', 'russell', 'alonso', 'stroll', 'bottas'
]

# Filter for drivers actually in the affinity data
available_drivers = [d for d in top_drivers if d in affinity_df['driver_id'].values]
print(f'Drivers with affinity data: {len(available_drivers)}')
for d in available_drivers:
    n = (affinity_df['driver_id'] == d).sum()
    print(f'  {d}: {n} circuits')

fig = affinity_heatmap(affinity_df, drivers=available_drivers)
fig.update_layout(title_text='Driver-Circuit Affinity Heatmap (Top 2024 Drivers)')
fig.show()

## 3. Top Circuits Per Driver

Ranked bar charts showing which circuits each driver performs best at.

In [ ]:
# VER: Top circuits
ver_driver_id = 'max_verstappen'
if ver_driver_id in affinity_df['driver_id'].values:
    fig = top_circuits_bar(affinity_df, ver_driver_id, top_n=10)
    fig.update_layout(title_text='Verstappen: Top 10 Circuits by Affinity Score')
    fig.show()
else:
    print(f'No affinity data for {ver_driver_id}')

In [ ]:
# HAM: Top circuits
ham_driver_id = 'hamilton'
if ham_driver_id in affinity_df['driver_id'].values:
    fig = top_circuits_bar(affinity_df, ham_driver_id, top_n=10)
    fig.update_layout(title_text='Hamilton: Top 10 Circuits by Affinity Score')
    fig.show()
else:
    print(f'No affinity data for {ham_driver_id}')

## 4. Circuit Type Analysis

Circuits are classified into types: **street**, **high_speed**, **technical**, and **mixed**.
Different teams and drivers have strengths aligned to specific circuit types.

In [ ]:
# Circuit types from the database
circuit_types = con.execute("""
    SELECT ct.circuit_type, COUNT(*) AS circuits,
           GROUP_CONCAT(DISTINCT ct.circuit_id) AS circuit_list
    FROM circuit_types ct
    GROUP BY ct.circuit_type
    ORDER BY circuits DESC
""").fetchdf()

print('Circuit type classification:')
display(circuit_types)

In [ ]:
# Average affinity score by circuit type for top drivers
type_affinity = con.execute("""
    SELECT
        a.driver_id,
        ct.circuit_type,
        ROUND(AVG(a.affinity_score), 1) AS avg_affinity,
        COUNT(*) AS circuits
    FROM driver_circuit_affinity a
    JOIN circuit_types ct ON a.circuit_id = ct.circuit_id
    WHERE a.affinity_score IS NOT NULL
    GROUP BY a.driver_id, ct.circuit_type
    HAVING COUNT(*) >= 2
    ORDER BY a.driver_id, avg_affinity DESC
""").fetchdf()

if not type_affinity.empty:
    # Pivot for visualization
    pivot = type_affinity.pivot_table(
        index='driver_id', columns='circuit_type', values='avg_affinity'
    )
    
    # Filter to drivers with data across multiple types
    pivot = pivot.dropna(thresh=2)
    if len(pivot) > 10:
        # Keep top drivers by mean affinity
        pivot = pivot.loc[pivot.mean(axis=1).nlargest(10).index]
    
    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=list(pivot.columns),
        y=list(pivot.index),
        colorscale=[[0, '#1a1a2e'], [0.5, '#e94560'], [1, '#00FF00']],
        text=pivot.values.round(1),
        texttemplate='%{text}',
    ))
    fig = f1_layout(fig, 'Driver Affinity by Circuit Type', height=max(400, len(pivot) * 35))
    fig.show()
else:
    print('No circuit type affinity data available')

## 5. Case Study: Hamilton at Silverstone

Lewis Hamilton's record at the British Grand Prix is legendary. We deep-dive into his
Silverstone performance history to quantify this dominance.

In [ ]:
# Hamilton's Silverstone record
ham_silverstone = con.execute("""
    SELECT
        r.year,
        ra.race_name,
        r.grid,
        r.position,
        r.points,
        r.status,
        r.grid - r.position AS positions_gained
    FROM results r
    JOIN races ra ON r.race_id = ra.race_id
    WHERE r.driver_id = 'hamilton'
      AND ra.circuit_id = 'silverstone'
    ORDER BY r.year
""").fetchdf()

if not ham_silverstone.empty:
    wins = (ham_silverstone['position'] == 1).sum()
    podiums = (ham_silverstone['position'] <= 3).sum()
    avg_finish = ham_silverstone['position'].dropna().mean()
    
    print(f'Hamilton at Silverstone: {len(ham_silverstone)} races')
    print(f'  Wins: {wins}')
    print(f'  Podiums: {podiums}')
    print(f'  Average finish: {avg_finish:.1f}')
    print(f'  Total points: {ham_silverstone["points"].sum()}')
    display(ham_silverstone)
else:
    print('No Hamilton Silverstone data found')

In [ ]:
# Visualize Hamilton's Silverstone finishing positions over time
if not ham_silverstone.empty:
    fig = go.Figure()
    
    # Color code by podium/non-podium
    colors = ['#FFD700' if p == 1 else '#C0C0C0' if p == 2 else '#CD7F32' if p == 3 else '#666'
              for p in ham_silverstone['position'].fillna(20)]
    
    fig.add_trace(go.Bar(
        x=ham_silverstone['year'],
        y=ham_silverstone['position'],
        marker_color=colors,
        text=ham_silverstone['position'].astype(str),
        textposition='outside'
    ))
    
    fig = f1_layout(fig, 'Hamilton at Silverstone: Finishing Positions', height=400)
    fig.update_yaxes(title_text='Finishing Position', autorange='reversed', range=[0.5, 20.5])
    fig.update_xaxes(title_text='Year')
    fig.add_hline(y=3.5, line_dash='dash', line_color='#FFD700', annotation_text='Podium line')
    fig.show()

## 6. Driver History at a Circuit

Looking at the driver-circuit history view, which captures aggregate performance
metrics per driver at each circuit.

In [ ]:
# Top performers at Silverstone historically
silverstone_kings = con.execute("""
    SELECT
        dch.driver_id,
        dch.circuit_id,
        dch.total_appearances,
        ROUND(dch.avg_finish_position, 2) AS avg_finish,
        ROUND(dch.finish_pct_rank, 3) AS finish_pct_rank,
        ROUND(dch.positions_gained_avg, 2) AS positions_gained_avg,
        ROUND(dch.dnf_rate, 3) AS dnf_rate
    FROM driver_circuit_history dch
    WHERE dch.circuit_id = 'silverstone'
      AND dch.total_appearances >= 3
    ORDER BY dch.avg_finish_position
    LIMIT 15
""").fetchdf()

if not silverstone_kings.empty:
    print('Top performers at Silverstone (min 3 races):')
    display(silverstone_kings)
    
    fig = go.Figure(go.Bar(
        x=silverstone_kings['avg_finish'],
        y=silverstone_kings['driver_id'],
        orientation='h',
        marker_color=F1_RED,
        text=silverstone_kings['avg_finish'].round(1),
        textposition='outside'
    ))
    fig = f1_layout(fig, 'Silverstone: Best Average Finishing Positions', height=500)
    fig.update_xaxes(title_text='Average Finishing Position', autorange='reversed')
    fig.show()

## 7. Circuit Similarity Clustering

We can compare how similarly different drivers perform across circuits to identify
circuits that share DNA: if a driver excels at Circuit A and Circuit B, those circuits
may reward similar driving characteristics.

In [ ]:
# Build correlation matrix of circuits based on driver affinity scores
if not affinity_df.empty:
    circuit_pivot = affinity_df.pivot_table(
        index='driver_id', columns='circuit_id', values='affinity_score'
    )
    
    # Only keep circuits with enough data
    circuit_pivot = circuit_pivot.dropna(thresh=5, axis=1)  # at least 5 drivers
    circuit_pivot = circuit_pivot.dropna(thresh=3, axis=0)  # at least 3 circuits per driver
    
    if circuit_pivot.shape[1] >= 3:
        corr = circuit_pivot.corr()
        
        fig = go.Figure(go.Heatmap(
            z=corr.values,
            x=list(corr.columns),
            y=list(corr.index),
            colorscale='RdBu_r',
            zmin=-1, zmax=1,
            text=corr.values.round(2),
            texttemplate='%{text}',
        ))
        fig = f1_layout(fig, 'Circuit Similarity Matrix (Driver Affinity Correlation)',
                        height=max(500, len(corr) * 30))
        fig.show()
        
        print(f'\nCircuits analyzed: {len(corr)}')
    else:
        print(f'Insufficient circuit data for correlation (need 3+, have {circuit_pivot.shape[1]})')
else:
    print('No affinity data for circuit similarity analysis')

## Key Findings

1. **Driver-circuit affinity** is a strong predictor of race outcomes. Drivers with high affinity scores at a circuit consistently outperform their season average there.
2. **Hamilton at Silverstone** is the poster child for circuit affinity, with a record number of wins at a single venue.
3. **Circuit types** cluster performance: drivers who excel on street circuits tend to underperform on high-speed circuits, and vice versa.
4. **Circuit similarity** reveals hidden connections: e.g., Spa and Silverstone show correlated performance patterns due to their high-speed nature.

---
*Next: Module D (Constructor Efficiency) analyzes team-level performance relative to expectations.*

In [ ]:
con.close()
print('Connection closed.')